# RG3 가스: 닮은 정상 샷 가중 학습

가스와 가장 가까운 정상20%를 각 학습ref의 순위75 공간에서 선정. 정상 가중3/10, 기본rank75, std4 전문가를 내부 F1 기준 단독/쌍평균 선택. 평가 가스/정상을 hard mining에 사용하지 않음. 외부/내부 모델마다 ref별 정상척도·가스거리·정상선정·분류·임계값 학습. 각 전문가는 RF/ET2시드 평균. RG3 RH620(가스19/정상601), 같은7개 관측구간 중첩검증. 새 독립 검증 아님.


In [1]:
from pathlib import Path
import numpy as np,pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.ensemble import RandomForestClassifier,ExtraTreesClassifier
from sklearn.metrics import confusion_matrix,precision_recall_curve,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
t=root/'reports/tables'
z=np.load(t/'rg3_F1_boost_fold_cache.npz',allow_pickle=False)
y=z['labels'];groups=z['groups'];times=z['timestamps'];blocks=np.unique(groups)
assert len(y)==620 and y.sum()==19 and (y==0).sum()==601
sources={tuple(z[k].astype(int)):k.split('_')[1] for k in z.files if k.startswith('ref_')}
memo={};audit=[]
def predict(ref,q):
 assert set(groups[ref]).isdisjoint(groups[q])
 key=tuple(ref)
 if key not in memo:
  n=sources[key];F=z['features_'+n+'_1'];old=z['scores_'+n]
  normal=np.flatnonzero(y[ref]==0);positive=np.flatnonzero(y[ref]==1)
  scale=RobustScaler().fit(F[ref[normal]])
  tr=scale.transform(F[ref])
  distance=NearestNeighbors(n_neighbors=1).fit(tr[positive]).kneighbors(tr[normal],return_distance=True)[0][:,0]
  count=max(1,int(np.ceil(.2*len(normal))));hard=normal[np.argsort(distance,kind='stable')[:count]]
  values=[old[:,1]]
  for weight in [3.,10.]:
   w=np.ones(len(ref));w[hard]=weight;preds=[]
   for cls in [RandomForestClassifier,ExtraTreesClassifier]:
    for seed in [42,7]:
     m=cls(n_estimators=80,max_depth=4,min_samples_leaf=4,class_weight='balanced',random_state=seed,n_jobs=2)
     preds.append(m.fit(F[ref],y[ref],sample_weight=w).predict_proba(F)[:,1])
   values.append(np.mean(preds,axis=0))
  values.append(old[:,0])
  memo[key]=np.column_stack(values)
  audit.append(dict(train=len(ref),gas_training=len(positive),hard_normal_training=len(hard),train_blocks=','.join(map(str,np.unique(groups[ref]))),excluded_blocks=','.join(map(str,np.setdiff1d(blocks,groups[ref])))))
 return memo[key][q]
def stats(yy,p):
 tn,fp,fn,tp=confusion_matrix(yy,p,labels=[0,1]).ravel()
 return dict(TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,recall=tp/(tp+fn) if tp+fn else 0)
def cutfit(yy,score):
 p,r,c=precision_recall_curve(yy,score);f=np.divide(2*p[:-1]*r[:-1],p[:-1]+r[:-1],out=np.zeros_like(c),where=p[:-1]+r[:-1]>0)
 cut=float(c[np.lexsort((c,r[:-1],f))[-1]]);return cut,stats(yy,score>=cut)
weights=[np.eye(4)[i] for i in range(4)]+[(np.eye(4)[i]+np.eye(4)[j])/2 for i in range(4) for j in range(i+1,4)]
records=[];choices=[]
for b in blocks:
 q=np.flatnonzero(groups==b);ref=np.flatnonzero(groups!=b)
 out=predict(ref,q);inner=np.full((len(ref),4),np.nan)
 for ib in np.unique(groups[ref]):
  mask=groups[ref]==ib;inner[mask]=predict(ref[~mask],ref[mask])
 candidates=[]
 for w in weights:
  cut,result=cutfit(y[ref],inner@w);candidates.append((result['F1'],result['recall'],-result['FP'],w,cut,out@w))
 best=max(candidates,key=lambda a:a[:3])
 named=[('inner_F1_selected_hard_negative',best[5],best[4])]
 for i,name in enumerate(['rank75_reference','hard_normal_weight3','hard_normal_weight10','deep_std4_reference']):
  cut,_=cutfit(y[ref],inner[:,i]);named.append((name+'_inner_F1_cut',out[:,i],cut))
 for name,score,cut in named:
  for j,i in enumerate(q):records.append(dict(model=name,TimeStamp=times[i],block=int(b),actual=int(y[i]),prediction=int(score[j]>=cut),score=float(score[j]),threshold=cut))
 choices.append(dict(block=int(b),weights=','.join(map(str,best[3])),threshold=best[4],inner_F1=best[0]))
 (t/'rg3_hard_negative_progress.json').write_text(str(int(b)),encoding='utf-8')
 print('Finished hard-negative block',b,flush=True)
p=pd.DataFrame(records);m=pd.DataFrame([dict(model=name,AP=average_precision_score(g.actual,g.score),**stats(g.actual,g.prediction)) for name,g in p.groupby('model')])
for obj,name in [(p,'predictions'),(m,'metrics'),(pd.DataFrame(choices),'inner_choices'),(pd.DataFrame(audit),'training_audit')]:obj.to_csv(t/f'rg3_hard_negative_{name}.csv',index=False,encoding='utf-8-sig')
print(m.to_string(index=False))


Finished hard-negative block 1


Finished hard-negative block 2


Finished hard-negative block 3


Finished hard-negative block 4


Finished hard-negative block 5


Finished hard-negative block 6


Finished hard-negative block 7


                            model       AP  TP  FP  FN  TN       F1   recall
 deep_std4_reference_inner_F1_cut 0.079209   4  50  15 551 0.109589 0.210526
hard_normal_weight10_inner_F1_cut 0.065609   4  34  15 567 0.140351 0.210526
 hard_normal_weight3_inner_F1_cut 0.074648   5  36  14 565 0.166667 0.263158
  inner_F1_selected_hard_negative 0.073583   5  38  14 563 0.161290 0.263158
    rank75_reference_inner_F1_cut 0.076694   5  35  14 566 0.169492 0.263158
